# Predicting High-Risk Drug Interactions in FAERS
## Full KDD Pipeline Notebook

**CS 584 — Data Mining · Illinois Institute of Technology · Spring 2026**
Sergio Illescas Cabiró · Carmen Vázquez Pérez de la Cruz · Laura Rueda García

---

This notebook runs the **complete Knowledge Discovery in Databases (KDD) pipeline** end-to-end:

| Phase | Steps | Output |
|---|---|---|
| **Task A — Interaction Mining** | Ingestion → Preprocessing → Encoding → Apriori / FP-Growth → Filter → Plots → Metrics | 13 high-lift drug-interaction rules |
| **Task B — Severity Prediction** | Ingestion → Feature engineering → 5 classifiers → Optuna tuning | AUC-ROC 0.8054 (tuned ensemble) |

**How this notebook works**: each cell calls the corresponding Python script via `subprocess`.
Scripts are the single source of truth — the notebook orchestrates, displays results, and explains the methodology.


## Setup

In [ ]:
import os, sys

# Auto-detect project root: works whether you open from root or from taskB/
if not os.path.isdir("taskA"):
    parent = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(parent, "taskA")):
        os.chdir(parent)
    else:
        raise RuntimeError("Cannot find project root. Open notebook from DrugInteractionsRisks/")

print("Working directory:", os.getcwd())


In [ ]:
import subprocess, sys, os, textwrap

def run(label, cmd):
    """Run a shell command, stream output, raise on failure."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True, encoding="utf-8"
    )
    if result.stdout:
        print(result.stdout[-4000:])   # last 4000 chars to avoid flooding
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"Step failed: {label}")
    print(f"  ✓ Done: {label}")
    return result


In [ ]:
import importlib

REQUIRED = ["pandas", "pyarrow", "mlxtend", "networkx", "matplotlib",
            "sklearn", "xgboost", "catboost", "optuna", "tqdm"]

missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing missing packages:", missing)
    run("pip install", f"pip install {' '.join(missing)}")
else:
    print("All required packages are installed.")


---
## Task A — Association Rule Mining

We apply two classic frequent-itemset algorithms (**Apriori** and **FP-Growth**) to the raw FDA Adverse Event Reporting System (FAERS) data to discover which drug combinations appear together with adverse reactions far more often than expected by chance.

### Pipeline overview

```
FAERS JSON files
  → 1A  Ingestion          → consolidated_data.parquet  (108k+ reports)
  → 2A  Preprocessing      → transaction baskets         (drug + reaction)
  → 3A  Encoding           → binary one-hot matrix
  → 4A  Apriori            → association rules (support, confidence, lift)
  → 5A  FP-Growth          → same rules (cross-validation of correctness)
  → 6A  Indication filter  → removes therapeutic-use noise
  → 7A  Network plots      → drug-reaction graphs
  → 8A  Metrics table      → final_metrics_table.csv
```


### Step 1A — Data Ingestion

Reads the 9 raw FAERS quarterly JSON files, flattens the nested structure, and saves a consolidated Parquet file.

**Key fields extracted**: `safetyreportid`, `patient.drug.activesubstance.activesubstancename`, `patient.reaction.reactionmeddrapt`, `patient.patientsex`, `patient.patientonsetage`, `serious*` fields.

> **Note**: FAERS JSON has two formats for `activesubstance` — sometimes a dict, sometimes a list. The ingestion script handles both.


In [ ]:
# 1A — requires the raw JSON folder path as argument
# Adjust the path to where your FAERS JSON files are located
JSON_FOLDER = "taskA/data"   # ← change if needed

if os.path.exists("consolidated_data.parquet"):
    print("consolidated_data.parquet already exists — skipping ingestion.")
else:
    run("1A Ingestion", f"python taskA/1A_dataIngestion.py {JSON_FOLDER}")


### Step 2A — Preprocessing

Extracts drug-reaction transaction baskets from the consolidated data.
Each report becomes a **basket** containing the active substance names and the MedDRA reaction terms reported together.

Two basket types are created:
- **Active substances only** — for drug-drug co-occurrence patterns
- **Drugs + reactions combined** — for discovering drug→reaction rules


In [ ]:
run("2A Preprocessing", "python taskA/2A_preprocessing.py")


### Step 3A — One-Hot Encoding

Converts transaction baskets into a **binary matrix** where each row is a report and each column is a drug or reaction term.
This is the input format required by both Apriori and FP-Growth.

> Example: row `[1, 0, 1, 0, ...]` means the report contains drug 1 and drug 3 but not drugs 2 or 4.


In [ ]:
run("3A Encoding", "python taskA/3A_Encoding.py")


### Steps 4A & 5A — Apriori and FP-Growth Rule Mining

Both algorithms find **frequent itemsets** (drug/reaction combinations appearing together in ≥ 0.25% of reports) and then derive **association rules** from them.

**Validated parameters**:
- `min_support = 0.0025` (0.25%) — empirically the lowest threshold where both algorithms terminate without MemoryError and produce interpretable rules
- Rules are evaluated by **support**, **confidence**, and **lift**

**Why run both?**
FP-Growth is typically faster on large datasets. Running both and getting identical results **cross-validates correctness**.


In [ ]:
ENCODED = "taskA/active_substances_encoded.parquet"
SUPPORT = 0.0025

run("4A Apriori",   f"python taskA/4A_APriori.py --file_path {ENCODED} --min_support {SUPPORT}")
run("5A FP-Growth", f"python taskA/5A_FPGrowth.py --file_path {ENCODED} --min_support {SUPPORT}")


### Step 6A — Indication Bias Filter

Raw rules include **therapeutic-use artifacts** like:
- `METHOTREXATE → RHEUMATOID ARTHRITIS` (Methotrexate is *prescribed for* RA, not causing it)
- `INSULIN → DIABETES` (insulin treats diabetes, not an adverse interaction)

This step removes rules where the consequent is a known indication for the antecedent drug, leaving only **true adverse drug interactions**.


In [ ]:
import glob

apriori_csvs  = glob.glob("taskA/association_rules/association_rules_APRIORI_*.csv")
fpgrowth_csvs = glob.glob("taskA/association_rules/association_rules_FPGROWTH_*.csv")

for csv_path in apriori_csvs + fpgrowth_csvs:
    run(f"6A Filter → {os.path.basename(csv_path)}",
        f"python taskA/6A_FilterTrueInteractions.py --input {csv_path}")


### Step 7A — Network Graph Visualization

Renders the filtered rules as **drug-reaction network graphs** using NetworkX.

- **Teal nodes** = drugs
- **Coral/pink nodes** = adverse reactions
- **Edge thickness** = proportional to lift (stronger signal = thicker edge)

This makes it easy to spot clusters of drugs sharing common adverse reactions.


In [ ]:
filtered_csvs = glob.glob("taskA/filtered_association_rules/filtered_*.csv")

for csv_path in filtered_csvs:
    run(f"7A Plots → {os.path.basename(csv_path)}",
        f"python taskA/7A_AnalysisPlots.py --file {csv_path}")


### Step 8A — Metrics Table

Generates the final evaluation metrics table summarizing the mining results.


In [ ]:
run("8A Metrics Table", "python taskA/8A_GenerateMetricsTable.py")


### Task A Results


In [ ]:
import pandas as pd
from IPython.display import display, Image

# Final metrics
metrics_path = "taskA/evaluation/final_metrics_table.csv"
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print("Task A — Final Metrics")
    display(metrics_df)
else:
    print(f"Metrics table not found at {metrics_path}")

# Network plots
plot_paths = sorted(glob.glob("taskA/TaskAPlots/*.png"))
for p in plot_paths:
    print(f"\n{os.path.basename(p)}")
    display(Image(p))


#### Interpretation

Both Apriori and FP-Growth produce **13 identical rules** at `min_support=0.0025` — confirming algorithmic correctness.

| Metric | Value |
|---|---|
| Rules found | **13** |
| Max Lift | **223.45** (LEUCOVORIN + FLUOROURACIL → DIARRHEA) |
| Avg Confidence | 0.637 |

A lift of **223×** means LEUCOVORIN + FLUOROURACIL co-occur with diarrhea 223 times more often than expected by chance — a clinically well-known interaction (FOLFOX chemotherapy regimen).


---
## Bridge: From Rules to Features

Task A's association rules feed directly into Task B as predictive features.

For each patient report, `2B_preprocessing.py` checks whether the drugs/reactions in that report **match any of the 13 high-lift rules** (lift ≥ 2.0). If yes, a binary feature `has_drug_reaction_rule = 1` is set.

This feature became **the #1 most predictive feature** in the tuned XGBoost classifier — outperforming age, sex, number of drugs, and all other engineered features.

> This is the core KDD loop: domain knowledge from unsupervised mining (Task A) becomes a supervised signal (Task B).


---
## Task B — Severity Prediction

Binary classification: predict whether a new FAERS report will have a **severe outcome** (death, hospitalization, or life-threatening reaction).

### Pipeline overview

```
consolidated_data.parquet  (shared with Task A)
  → 1B  Re-ingestion       → ensures ZIP source is processed (optional if Task A ran)
  → 2B  Feature engineering → task_b_features.parquet  (71 features)
  → 3B  Modeling           → 5 classifiers + threshold tuning + ROC/PR curves
  → 4B  Optuna tuning      → Bayesian HPO for XGBoost + CatBoost (GPU)
```

### Feature matrix (71 features)

| Feature group | Examples |
|---|---|
| Patient demographics | `patientonsetage`, `patientsex` |
| Report characteristics | `num_drugs`, `num_reactions`, `report_quarter` |
| Drug-level flags | `has_drug_reaction_rule` (from Task A), `num_suspect_drugs` |
| Reaction MedDRA encoding | Top-50 reaction one-hot flags |
| Interaction indicators | `max_rule_lift`, `num_matching_rules` |


### Step 1B — Data Ingestion (Task B)

Reads the FAERS ZIP archive containing all 9 quarterly JSON files and produces `consolidated_data.parquet`.
If `consolidated_data.parquet` already exists from Task A, this step can be skipped.


In [ ]:
ZIP_PATH = "taskB/9 json files - no tocar.zip"   # ← adjust if needed

if os.path.exists("consolidated_data.parquet"):
    print("consolidated_data.parquet already exists — skipping Task B ingestion.")
else:
    run("1B Ingestion", f'python taskB/1B_dataIngestion.py "{ZIP_PATH}"')


### Step 2B — Feature Engineering

Builds the 71-feature classification matrix from `consolidated_data.parquet`.

Key engineering decisions:
- `patientonsetage` NaN values are **preserved** — imputed inside each model's `sklearn.Pipeline` to prevent data leakage
- `rxn_hospitalisation` (MedDRA code) was removed after detecting it **directly encoded the target** (AUC 0.805→0.796 drop confirmed leakage)
- Task A rules are matched here: `has_drug_reaction_rule`, `max_rule_lift`, `num_matching_rules`


In [ ]:
run("2B Feature Engineering", "python taskB/2B_preprocessing.py")


In [ ]:
# Quick EDA of the feature matrix
features_df = pd.read_csv("taskB/task_b_features.parquet") if False else None
try:
    features_df = pd.read_parquet("taskB/task_b_features.parquet")
    print(f"Feature matrix shape: {features_df.shape}")
    print(f"Severe outcome rate: {features_df['severe'].mean():.1%}")
    print("\nFeature types:")
    print(features_df.dtypes.value_counts())
    display(features_df.head(3))
except FileNotFoundError:
    print("task_b_features.parquet not found — run 2B preprocessing first.")


### Step 3B — Baseline Modeling

Trains 5 classifiers with stratified 70/10/20 train/val/test split:

| Model | Notes |
|---|---|
| Logistic Regression | L2 regularized, baseline linear model |
| Random Forest | 500 trees, `class_weight="balanced"` |
| XGBoost (GPU) | `scale_pos_weight` for class imbalance |
| CatBoost (GPU) | `auto_class_weights="Balanced"` |
| Voting Ensemble | Soft voting: RF + XGBoost + CatBoost |

**Threshold tuning**: the classification threshold is optimized on the validation set to maximize F1-severe (not default 0.5, which is suboptimal for imbalanced data).


In [ ]:
run("3B Modeling", "python taskB/3B_modeling.py")


In [ ]:
# Display baseline results
eval_path = "taskB/task_b_evaluation.csv"
if os.path.exists(eval_path):
    eval_df = pd.read_csv(eval_path)
    print("Baseline Model Results")
    display(eval_df.style.highlight_max(subset=["AUC-ROC", "F1 (severe=1)", "Recall (severe)"],
                                        color="lightgreen"))
else:
    print(f"Results not found at {eval_path}")

# ROC curves
for plot_name in ["roc_curves.png", "confusion_matrices.png", "feature_importance_rf.png"]:
    p = f"taskB/TaskBPlots/{plot_name}"
    if os.path.exists(p):
        print(f"\n{plot_name}")
        display(Image(p))


### Step 4B — Optuna Hyperparameter Tuning

Bayesian hyperparameter optimization using Optuna's **TPE sampler**:

| Model | Trials | Search space highlights |
|---|---|---|
| XGBoost | 100 | learning_rate, max_depth, subsample, colsample_bytree, reg_alpha, reg_lambda |
| CatBoost | 50 | learning_rate, depth, l2_leaf_reg, bagging_temperature, random_strength |

Both use **early stopping** (patience=50) against the validation set AUC, so `n_estimators` is automatically determined.

> Runtime: approximately 18–22 minutes on an NVIDIA RTX 3060 GPU.


In [ ]:
run("4B Optuna Tuning (this may take ~20 minutes on GPU)", "python taskB/4B_optuna_tuning.py")


In [ ]:
import json

# Tuned results
tuned_path = "taskB/task_b_tuned_evaluation.csv"
if os.path.exists(tuned_path):
    tuned_df = pd.read_csv(tuned_path)
    print("Tuned Model Results")
    display(tuned_df.style.highlight_max(subset=["AUC-ROC", "F1 (severe=1)", "Recall (severe)"],
                                         color="lightgreen"))

# Best params
params_path = "taskB/optuna_best_params.json"
if os.path.exists(params_path):
    with open(params_path) as f:
        best_params = json.load(f)
    print("\nBest XGBoost params:")
    print(json.dumps(best_params.get("xgboost", {}), indent=2))
    print("\nBest CatBoost params:")
    print(json.dumps(best_params.get("catboost", {}), indent=2))

# Optimization history plots
for plot_name in ["optuna_history_xgb.png", "optuna_history_catboost.png",
                  "roc_curves_tuned.png", "feature_importance_xgb_tuned.png"]:
    p = f"taskB/TaskBPlots/{plot_name}"
    if os.path.exists(p):
        print(f"\n{plot_name}")
        display(Image(p))


---
## Conclusions

### Task A — What we found

- **13 high-quality drug-interaction rules** survive the indication-bias filter
- The strongest: `LEUCOVORIN + FLUOROURACIL → DIARRHEA` (lift = **223.45×**)
- Apriori and FP-Growth produce **identical results** — mutual validation of algorithmic correctness
- `min_support = 0.0025` is the empirically validated sweet spot: lower values cause MemoryError or produce 350k+ noisy rules

### Task B — What we achieved

| Model | AUC-ROC | F1-severe | Recall-severe |
|---|---|---|---|
| Logistic Regression | 0.7519 | 0.533 | 0.641 |
| Random Forest | 0.7916 | 0.570 | 0.644 |
| XGBoost (GPU) | 0.7949 | 0.574 | 0.685 |
| CatBoost (GPU) | 0.7936 | 0.574 | 0.641 |
| Voting Ensemble | 0.7962 | 0.575 | 0.648 |
| **Tuned Ensemble (Optuna)** | **0.8054** | **0.586** | **0.651** |

### Key insight: Task A rules as Task B features

The feature `has_drug_reaction_rule` (binary flag: does this report match a high-lift rule from Task A?) became the **#1 most predictive feature** in the tuned XGBoost model — outperforming age, sex, number of drugs, and all engineered features.

This validates the KDD pipeline design: **unsupervised pattern discovery (Task A) directly improves supervised prediction (Task B)**.

### Leakage detection

During development, `rxn_hospitalisation` (a MedDRA reaction code) was removed after it caused a suspicious AUC of 0.862. Removing it dropped AUC from 0.805 → 0.796, confirming real data leakage (the feature directly encoded the target). Optuna tuning on the clean feature set recovered to **AUC 0.8054**.
